# 05 — LLM-only vs LLM+RAG Comparison

Loads saved metrics from:
- `research/results/llm_baseline_metrics.json`
- `research/results/rag_metrics.json`

Produces comparison tables/charts for the dissertation discussion.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "research").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "research"))

from compare_rag import write_comparison_artifacts

RESULTS = ROOT / "research" / "results"
FIGURES = ROOT / "research" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

summary = write_comparison_artifacts(
    baseline_metrics_path=RESULTS / "llm_baseline_metrics.json",
    rag_metrics_path=RESULTS / "rag_metrics.json",
    rag_predictions_path=RESULTS / "rag_predictions.csv",
    out_dir=RESULTS,
)
table = pd.DataFrame(summary["comparison_table"])
display(table)
print("Retrieval stats:", json.dumps(summary.get("retrieval_stats", {}), indent=2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(table))
width = 0.35
ax.bar([i - width / 2 for i in x], table["llm_only"], width, label="LLM-only")
ax.bar([i + width / 2 for i in x], table["llm_rag"], width, label="LLM+RAG")
ax.set_xticks(list(x))
ax.set_xticklabels(table["metric"], rotation=20, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("LLM-only vs LLM+RAG")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "llm_vs_rag_metrics.png", dpi=150)
plt.show()

## Discussion prompts (dissertation)

1. **Reliability:** Did accuracy / macro-F1 rise, fall, or stay flat? Which classes improved?
2. **Trustworthiness:** Do RAG explanations cite NHS/Student Minds/Samaritans sources more often?
3. **Explainability:** Compare reasoning length and presence of `retrieved_sources`.
4. **Failure modes:** Cases where retrieval was off-topic and hurt the prediction.
5. **Limits:** Pending KB docs, Mind 403 gaps, SWMH Reddit domain vs NHS guidance domain shift.